<a href="https://colab.research.google.com/github/AlperYildirim1/geometric-grokking/blob/main/Final_No_attention_all_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import numpy as np
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# GOOGLE DRIVE SETUP
# ==========================================
SAVE_DIR = '/content/drive/MyDrive/Grokking_Zero_Attention_Results_All_Models_1e-4'
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# CONFIGURATION
# ==========================================
P = 113
FRAC_TRAIN = 0.3
D_MODEL = 128
NUM_HEADS = 4
MLP_DIM = 512
LR = 1e-4           # Standard grokking LR
EPOCHS = 20000      # 20k steps is plenty to see if they fail or succeed
LOG_EVERY = 1000
SEEDS = list(range(1, 11)) # 10 seeds: 1 to 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# BULLETPROOF DETERMINISM
# ==========================================
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ==========================================
# DATASET GENERATION
# ==========================================
def make_dataset(p, frac_train, seed=42):
    set_seed(seed)
    rng = random.Random(seed)
    all_pairs = [(a, b) for a in range(p) for b in range(p)]
    rng.shuffle(all_pairs)

    n_train = int(len(all_pairs) * frac_train)
    train_x = torch.tensor([[a, b, p] for a, b in all_pairs[:n_train]], dtype=torch.long)
    train_y = torch.tensor([(a + b) % p for a, b in all_pairs[:n_train]], dtype=torch.long)
    test_x = torch.tensor([[a, b, p] for a, b in all_pairs[n_train:]], dtype=torch.long)
    test_y = torch.tensor([(a + b) % p for a, b in all_pairs[n_train:]], dtype=torch.long)
    return train_x, train_y, test_x, test_y

# ==========================================
# ABLATED UNIFIED TRANSFORMER
# ==========================================
class AblatedUnifiedTransformer(nn.Module):
    def __init__(self, p, d_model, num_heads, mlp_dim, model_type):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.model_type = model_type # 'no_norm', 'layernorm', 'spherical_wd', 'spherical_nowd'

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.mlp_in = nn.Linear(d_model, mlp_dim)
        self.mlp_out = nn.Linear(mlp_dim, d_model)

        self.unembed = nn.Linear(d_model, p, bias=False)

        if self.model_type == 'layernorm':
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)
            self.norm3 = nn.LayerNorm(d_model)

    def apply_norm(self, x, norm_module=None):
        if self.model_type == 'layernorm':
            return norm_module(x)
        elif 'spherical' in self.model_type:
            return F.normalize(x, dim=-1)
        return x # no_norm

    def forward(self, x):
        B, L = x.shape
        pos = torch.arange(L, device=x.device).unsqueeze(0)
        h = self.tok_embed(x) + self.pos_embed(pos)

        h = self.apply_norm(h, getattr(self, 'norm1', None))

        Q = self.W_Q(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_K(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_V(h).view(B, L, self.num_heads, self.d_head).transpose(1, 2)

        # ==========================================
        # HARD ABLATION OVERRIDE
        # Forces uniform attention routing [1/3, 1/3, 1/3]
        # ==========================================
        scores = torch.zeros((B, self.num_heads, L, L), device=x.device)

        attn_out = self.W_O((F.softmax(scores, dim=-1) @ V).transpose(1, 2).contiguous().view(B, L, self.d_model))

        h = h + attn_out
        h = self.apply_norm(h, getattr(self, 'norm2', None))

        h = h + self.mlp_out(F.relu(self.mlp_in(h)))
        h = self.apply_norm(h, getattr(self, 'norm3', None))

        h_final = h[:, 2, :]

        if self.model_type == 'spherical_nowd':
            # Fully Bounded
            w_normalized = F.normalize(self.unembed.weight, dim=1)
            logits = F.linear(h_final, w_normalized)
            return logits * 10.0
        else:
            return self.unembed(h_final)

# ==========================================
# TRAINING LOOP WITH HISTORY
# ==========================================
def train_model(model, train_x, train_y, test_x, test_y, epochs, weight_decay):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=weight_decay, betas=(0.9, 0.98))
    criterion = nn.CrossEntropyLoss()

    train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
    test_x, test_y = test_x.to(DEVICE), test_y.to(DEVICE)

    history = {'epoch': [], 'train_acc': [], 'test_acc': []}
    peak_test_acc = 0.0

    pbar = tqdm(range(epochs), desc="Training", leave=False)
    for epoch in pbar:
        model.train()
        logits = model(train_x)
        loss = criterion(logits, train_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % LOG_EVERY == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                train_acc = (logits.argmax(-1) == train_y).float().mean().item()
                test_logits = model(test_x)
                test_acc = (test_logits.argmax(-1) == test_y).float().mean().item()

                if test_acc > peak_test_acc:
                    peak_test_acc = test_acc

                history['epoch'].append(epoch)
                history['train_acc'].append(train_acc)
                history['test_acc'].append(test_acc)

            pbar.set_postfix({'tr_acc': f"{train_acc:.3f}", 'te_acc': f"{test_acc:.3f}"})

    return peak_test_acc, history

# ==========================================
# EXPERIMENT RUNNER & PLOTTING
# ==========================================
if __name__ == "__main__":
    results = []

    configs = [
        {"name": "No Norm Baseline", "type": "no_norm", "wd": 1.0},
        {"name": "LayerNorm Baseline", "type": "layernorm", "wd": 1.0},
        {"name": "Spherical Model (WD=1.0)", "type": "spherical_wd", "wd": 1.0},
        {"name": "Fully Bounded Sphere (WD=0.0)", "type": "spherical_nowd", "wd": 0.0}
    ]

    for cfg in configs:
        print(f"\n{'='*60}")
        print(f"=== Running Zero-Attention Ablation for: {cfg['name']} ===")
        print(f"{'='*60}")

        for seed in SEEDS:
            set_seed(seed)
            tr_x, tr_y, te_x, te_y = make_dataset(P, FRAC_TRAIN, seed=seed)

            model = AblatedUnifiedTransformer(P, D_MODEL, NUM_HEADS, MLP_DIM, model_type=cfg["type"]).to(DEVICE)

            peak_acc, history = train_model(model, tr_x, tr_y, te_x, te_y, EPOCHS, weight_decay=cfg["wd"])
            print(f"Seed {seed} Completed | Peak Test Acc: {peak_acc * 100:.2f}%")

            # --- Save the Plot to Drive ---
            plt.figure(figsize=(8, 5))
            plt.plot(history['epoch'], history['train_acc'], label='Train Acc', color='steelblue')
            plt.plot(history['epoch'], history['test_acc'], label='Test Acc', color='firebrick')
            plt.title(f"{cfg['name']} (Zero-Attention) - Seed {seed}")
            plt.xlabel("Epochs")
            plt.ylabel("Accuracy")
            plt.ylim(-0.05, 1.05)
            plt.legend()
            plt.grid(True, alpha=0.3)

            safe_name = cfg['name'].replace(' ', '_').replace('=', '').replace('(', '').replace(')', '')
            plot_filename = os.path.join(SAVE_DIR, f"{safe_name}_seed_{seed}.png")
            plt.savefig(plot_filename, dpi=150, bbox_inches='tight')
            plt.close()
            # ------------------------------

            results.append({
                "Architecture": cfg['name'],
                "Seed": seed,
                "Peak Test Accuracy": peak_acc
            })

    # Summary Dataframe
    print("\n" + "="*60)
    print(" 🚀 FINAL ZERO-ATTENTION ABLATION RESULTS 🚀")
    print("="*60 + "\n")
    df = pd.DataFrame(results)

    # Scale strictly for printing percentage
    summary_df = df.groupby("Architecture")["Peak Test Accuracy"].agg(
        Mean_Peak_Acc=lambda x: f"{x.mean()*100:.2f}%",
        Max_Peak_Acc=lambda x: f"{x.max()*100:.2f}%",
        Std_Dev=lambda x: f"{x.std()*100:.2f}%"
    ).reset_index()

    print(summary_df.to_markdown(index=False, tablefmt="github"))

    # Save Logs to Drive
    csv_path = os.path.join(SAVE_DIR, "zero_attention_ablation_results.csv")
    df.to_csv(csv_path, index=False)
    summary_csv = os.path.join(SAVE_DIR, "zero_attention_summary.csv")
    summary_df.to_csv(summary_csv, index=False)
    print(f"\n✅ All logs and charts saved successfully to: {SAVE_DIR}")